# 03 — Brown-dwarf test: fixed-mass 

Testing if we can correctly construct a star–planet–brown-dwarf system and reproduce its secular eccentricity evolution without stellar evolution for now

We use `SecularMultiple` through AMUSE. We have three physical bodies and two orbit records:


Outer orbit : Brown dwarf
Inner orbit: Star , Planet

Time starts at the beginning of this fixed-mass test, not at an age along the SeBa track.

The main test uses the masses similar to the final system we want to simulate but a deliberately much larger outer orbit of 20 AU. We first use the quadrupole approximation and compare with its analytical limit.

This is heavily based on the Amuse book 4.4.4 and the AMUSE repo SecularMultiple tests.


**This is the notebook where I had to do some heavy bug hunts so that is why you may see lots of run directories with .json and .csv and things like that popping up in the data directory. It helps me keep track of what is added and when. I'll try minimizing these in the future -Kostas**

In [ ]:
from pathlib import Path
from contextlib import contextmanager
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
from time import perf_counter
import csv
import json
import os
import platform
import subprocess
import sys
import uuid
import warnings
import zipfile

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from amuse.units import units, constants


def locate_project_root():
    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        for candidate in (parent, parent / "white_dwarf_project"):
            marker = candidate / "notebooks" / "01_stellar_evolution.ipynb"
            if marker.is_file():
                return candidate
    raise FileNotFoundError(
        f"Cannot find white_dwarf_project from {cwd}. "
        "Place this notebook beside 01_stellar_evolution.ipynb."
    )


@contextmanager
def working_directory(path):
    """Keep worker-generated files in this run's own directory. Restore on exit."""
    previous = Path.cwd()
    path = Path(path).resolve()
    path.mkdir(parents=True, exist_ok=True)
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not reported by package metadata"


def environment_record():
    result = {
        "python": sys.version,
        "python_executable": sys.executable,
        "platform": platform.platform(),
        "packages": {name: package_version(name) for name in (
            "amuse-framework", "amuse-seba", "amuse-secularmultiple",
            "amuse-huayno", "numpy", "matplotlib"
        )},
    }
    try:
        result["git_commit"] = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=ROOT,
            stderr=subprocess.DEVNULL, text=True, timeout=5
        ).strip()
        result["git_dirty"] = bool(subprocess.check_output(
            ["git", "status", "--porcelain"], cwd=ROOT,
            stderr=subprocess.DEVNULL, text=True, timeout=5
        ).strip())
    except (OSError, subprocess.SubprocessError):
        result["git_commit"] = None
        result["git_dirty"] = None
    return result


def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")


def write_rows_csv(path, rows):
    if not rows:
        raise ValueError("Cannot write an empty summary table.")
    with Path(path).open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


ROOT = locate_project_root()
DATA_DIR = ROOT / "data"
FIGURE_DIR = ROOT / "figures"

WORKER_OPTIONS = {}

print("Python executable:", sys.executable)
print("Project root:", ROOT)
print("AMUSE framework:", package_version("amuse-framework"))

from amuse.datamodel import Particles
from amuse.community.secularmultiple.interface import SecularMultiple

run_id = "dynamics_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
RUN_DIR = DATA_DIR / "validation_runs" / run_id
PLOT_DIR = FIGURE_DIR / "validation_runs" / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
PLOT_DIR.mkdir(parents=True, exist_ok=False)
save_json(RUN_DIR / "environment.json", environment_record())
print("SecularMultiple implementation:", SecularMultiple.__module__)
print("Run directory:", RUN_DIR)

## 1. Define the experiment

The masses `1.9 MEarth` and `27 MJupiter` are representative of the preferred  interpretation of KMT-2020-BLG-0414L in Zhang et al. 2024.

We use small nonzero eccentricities to avoid ambiguous circular-orbits and to encourage eccentricity change (**Putting them at 0 also did weird stuff to my code - Kostas**). The outer plane is tilted by an arbitrary 5 degrees to the coordinate reference plane, both nodes start at zero, so `i_inner = i_outer + I_mutual` gives the requested mutual inclination.

The duration is expressed in units of a characteristic Lidov–Kozai time. It is a scale, not the exact cycle period:

$
P_{\rm in}=2\pi\sqrt{\frac{a_{\rm in}^3}{G(m_\star+m_p)}},\quad
P_{\rm out}=2\pi\sqrt{\frac{a_{\rm out}^3}{G(m_\star+m_p+m_{\rm BD})}},$
$
t_{\rm LK}=\frac{8}{15\pi}\frac{m_\star+m_p+m_{\rm BD}}{m_{\rm BD}}
\frac{P_{\rm out}^2}{P_{\rm in}}(1-e_{\rm out}^2)^{3/2}.
$

The coefficient follows the characteristic timescale convention found in Antognini 2015 (**Is there a better timescale since then? This is the one I have used before - Kostas**). The orbit is measured relative to the inner pair's centre of mass.

In [ ]:
G_AU_MSUN_YR = float(constants.G.value_in(units.au**3 / (units.MSun * units.yr**2)))
M_PLANET_MSUN = float((1.9 | units.MEarth).value_in(units.MSun))
M_BD_MSUN = float((27.0 | units.MJupiter).value_in(units.MSun))
ENERGY_UNIT = units.MSun * units.au**2 / units.yr**2

BASE = {
    "m_star_MSun": 1.2,
    "m_planet_MSun": M_PLANET_MSUN,
    "m_bd_MSun": M_BD_MSUN,
    "a_in_AU": 1.0,
    "a_out_AU": 20.0,
    "e_in": 1.0e-3,
    "e_out": 1.0e-3,
    "mutual_inclination_deg": 60.0,
    "outer_reference_inclination_deg": 5.0,
    "omega_in_deg": 0.0,
    "omega_out_deg": 0.0,
    "node_in_deg": 0.0,
    "node_out_deg": 0.0,
    "order": "quadrupole",        # This can go to more depth e.g.: 'octupole', 'fifth'.
}
INCLINATIONS_DEG = (0.0, 30.0, 45.0, 60.0, 75.0, 120.0)
N_LK_TIMES = 20.0
OUTPUTS_PER_LK_TIME = 100
RTOL = 1e-13
ATOL_E = 1e-15
RUN_TARGET_SCALE_DEMO = True 
RUN_DIRECT_VALIDATION = True


def orbital_timescales(cfg):
    m_inner = cfg["m_star_MSun"] + cfg["m_planet_MSun"]
    m_total = m_inner + cfg["m_bd_MSun"]
    P_in = 2*np.pi*np.sqrt(cfg["a_in_AU"]**3 / (G_AU_MSUN_YR * m_inner))
    P_out = 2*np.pi*np.sqrt(cfg["a_out_AU"]**3 / (G_AU_MSUN_YR * m_total))
    t_lk = (8/(15*np.pi)) * (m_total/cfg["m_bd_MSun"]) * P_out**2/P_in * (1-cfg["e_out"]**2)**1.5
    return {"P_in_yr": float(P_in), "P_out_yr": float(P_out), "t_LK_yr": float(t_lk)}


def quadrupole_emax_limit(inclination_deg):
    """Near-circular inner test-particle limit"""
    value = 1.0 - (5.0/3.0)*np.cos(np.deg2rad(inclination_deg))**2
    return float(np.sqrt(max(0.0, value)))


def mutual_inclination_rad(i_in, i_out, node_in, node_out):
    cos_i = (np.cos(i_in)*np.cos(i_out) +
             np.sin(i_in)*np.sin(i_out)*np.cos(node_in-node_out))
    return np.arccos(np.clip(cos_i, -1.0, 1.0))


scales = orbital_timescales(BASE)
save_json(RUN_DIR / "experiment_settings.json", {
    "base": BASE, "inclinations_deg": list(INCLINATIONS_DEG),
    "number_LK_times": N_LK_TIMES, "outputs_per_LK_time": OUTPUTS_PER_LK_TIME,
    "rtol": RTOL, "atol_e": ATOL_E, "worker_options": WORKER_OPTIONS,
    "G_AU3_MSun_yr2": G_AU_MSUN_YR,
})
for name, value in scales.items():
    print(f"{name}: {value:.6g}")
print("Initial outer pericentre / inner semimajor axis:", BASE["a_out_AU"]*(1-BASE["e_out"])/BASE["a_in_AU"])
print("Analytical near-test-particle prediction at 60 degrees:", quadrupole_emax_limit(60.0))

## 2. Build the five AMUSE particle records

The first three entries are the star, planet, and brown dwarf. Entries 3 and 4 describe the inner and outer orbits. The outer orbit is (inner pair, brown dwarf), using the star alone as its inner child  creates the wrong hierarchy (**I tried, it confused me for a while -Kostas**).

AMUSE here takes ordinary floating-point angles in radians, whereas semimajor axes and masses use specifically AMUSE units.

The SecularMultiple implementation requires nonzero radii for its internal error weights. We use small positive placeholder radii on orbit records and positive representative radii on bodies. Radius dependent forces and collision checks are disabled, so these radii do not take much part in the dynamics.

In [ ]:
def make_hierarchy(cfg):
    masses = np.array([cfg["m_star_MSun"], cfg["m_planet_MSun"], cfg["m_bd_MSun"]])
    if not np.all(masses > 0):
        raise ValueError("All physical masses must be positive.")
    if cfg["a_out_AU"] <= cfg["a_in_AU"] or cfg["a_in_AU"] <= 0:
        raise ValueError("Require a_out > a_in > 0.")
    if not (0 < cfg["e_in"] < 1 and 0 < cfg["e_out"] < 1):
        raise ValueError("Use 0 < e < 1 in these benchmarks; use a small nonzero value for nearly circular orbits.")
    if cfg["node_in_deg"] != cfg["node_out_deg"]:
        raise ValueError("This constructor sets the mutual inclination using equal nodes; generalize it before changing nodes.")
    inner_inc = cfg["outer_reference_inclination_deg"] + cfg["mutual_inclination_deg"]
    if not 0 <= inner_inc <= 180:
        raise ValueError("This reference-plane choice requires 0 <= i_inner <= 180 degrees.")

    tree = Particles(5)
    tree.is_binary = [False, False, False, True, True]
    tree.child1 = None
    tree.child2 = None
    tree.mass = 0.0 | units.MSun
    tree[:3].mass = masses | units.MSun
    tree.radius = 1.0e-15 | units.au  # Nonzero placeholder for orbit records.
    tree.spin_vec_x = 0.0 | (1 / units.yr)
    tree.spin_vec_y = 0.0 | (1 / units.yr)
    tree.spin_vec_z = 0.0 | (1 / units.yr)
    tree[0].radius = 1.2 | units.RSun
    tree[1].radius = 1.2 | units.REarth
    tree[2].radius = 1.0 | units.RJupiter

    # Placed the initial orbital elements on the binary records, not the physical bodies.
    tree.semimajor_axis = 1.0 | units.au
    tree.eccentricity = 1.0e-3
    tree.inclination = 0.1
    tree.argument_of_pericenter = 0.0
    tree.longitude_of_ascending_node = 0.0

    inner, outer = tree[3], tree[4]
    inner.child1, inner.child2 = tree[0], tree[1]
    outer.child1, outer.child2 = inner, tree[2]
    inner.semimajor_axis = cfg["a_in_AU"] | units.au
    outer.semimajor_axis = cfg["a_out_AU"] | units.au
    inner.eccentricity, outer.eccentricity = cfg["e_in"], cfg["e_out"]
    inner.inclination = float(np.deg2rad(inner_inc))
    outer.inclination = float(np.deg2rad(cfg["outer_reference_inclination_deg"]))
    inner.argument_of_pericenter = float(np.deg2rad(cfg["omega_in_deg"]))
    outer.argument_of_pericenter = float(np.deg2rad(cfg["omega_out_deg"]))
    inner.longitude_of_ascending_node = float(np.deg2rad(cfg["node_in_deg"]))
    outer.longitude_of_ascending_node = float(np.deg2rad(cfg["node_out_deg"]))

    # Explicitly excluding extra physics because some things were enabled by default.
    for field in (
        "include_pairwise_1PN_terms", "include_pairwise_25PN_terms",
        "include_tidal_friction_terms", "include_tidal_bulges_precession_terms",
        "include_rotation_precession_terms", "check_for_secular_breakdown",
        "check_for_dynamical_instability", "check_for_physical_collision_or_orbit_crossing",
        "check_for_minimum_periapse_distance", "check_for_RLOF_at_pericentre",
        "sample_orbital_phases_randomly",
    ):
        setattr(tree, field, False)
    tree.mass_dot_external = 0.0 | (units.MSun / units.yr)
    tree.radius_dot_external = 0.0 | (units.au / units.yr)
    tree.radius_ddot_external = 0.0 | (units.au / units.yr**2)
    return tree


def configure_secular_solver(solver, order, rtol, atol_e):
    if order not in {"quadrupole", "octupole", "fifth"}:
        raise ValueError("Unknown multipole order.")
    solver.parameters.relative_tolerance = rtol
    solver.parameters.absolute_tolerance_eccentricity_vectors = atol_e
    solver.parameters.include_quadrupole_order_terms = True
    solver.parameters.include_octupole_order_binary_pair_terms = order in {"octupole", "fifth"}
    solver.parameters.include_octupole_order_binary_triplet_terms = False  # No triplet of binary orbits in a triple.
    solver.parameters.include_hexadecupole_order_binary_pair_terms = order == "fifth"
    solver.parameters.include_dotriacontupole_order_binary_pair_terms = order == "fifth"


tree = make_hierarchy(BASE)
print("Physical bodies:", tree[:3].mass.in_(units.MSun))
print("Inner orbit: star + planet")
print("Outer orbit: inner-pair centre of mass + brown dwarf")
print("Stored angles are radians; initial mutual inclination:",
      np.rad2deg(mutual_inclination_rad(tree[3].inclination, tree[4].inclination,
                                       tree[3].longitude_of_ascending_node, tree[4].longitude_of_ascending_node)))

## 3. One evolution routine

Our output times control the resolution of the saved eccentricity extremes. 

We read the orbital records held by the code, not the unchanged initial tree. Each saved state includes orbital elements, mutual inclination, and the secular Hamiltonian. An unexpected solver status, early return, or invalid eccentricity raises an error because some trouble happened with weird plots coming out.


In [ ]:
ORBIT_COLUMNS = ("a_in_AU", "e_in", "i_in_rad", "omega_in_rad", "node_in_rad",
                 "a_out_AU", "e_out", "i_out_rad", "omega_out_rad", "node_out_rad")


def read_secular_state(solver, remote):
    row = {
        "time_yr": float(solver.model_time.value_in(units.yr)),
        "hamiltonian": float(solver.hamiltonian.value_in(ENERGY_UNIT)),
    }

    # Read both orbits before validating the complete state.
    for suffix, binary in (("in", remote[3]), ("out", remote[4])):
        row[f"a_{suffix}_AU"] = float(
            binary.semimajor_axis.value_in(units.au)
        )
        row[f"e_{suffix}"] = float(binary.eccentricity)
        row[f"i_{suffix}_rad"] = float(binary.inclination)
        row[f"omega_{suffix}_rad"] = float(
            binary.argument_of_pericenter
        )
        row[f"node_{suffix}_rad"] = float(
            binary.longitude_of_ascending_node
        )

    # Remember to keep these checks outside the loop, had some trouble where they applied to jut one orbit.
    invalid = {
        name: value
        for name, value in row.items()
        if not np.isfinite(value)
    }
    if invalid:
        raise RuntimeError(
            f"Non-finite worker state at t={row['time_yr']} yr: {invalid}"
        )

    if not (0 <= row["e_in"] < 1 and 0 <= row["e_out"] < 1):
        raise RuntimeError(
            "Unbound/invalid eccentricity: "
            "this secular benchmark is no longer valid."
        )

    if row["a_in_AU"] <= 0 or row["a_out_AU"] <= 0:
        raise RuntimeError("Invalid semimajor axis.")

    # Derived diagnostics for any valid state.
    mutual = mutual_inclination_rad(
        row["i_in_rad"],
        row["i_out_rad"],
        row["node_in_rad"],
        row["node_out_rad"],
    )
    row["mutual_inclination_deg"] = float(np.rad2deg(mutual))
    row["q_in_AU"] = row["a_in_AU"] * (1 - row["e_in"])
    row["hierarchy_pericentre_ratio"] = (
        row["a_out_AU"] * (1 - row["e_out"]) / row["a_in_AU"]
    )

    return row


def run_secular(cfg, label, n_lk_times=N_LK_TIMES, outputs_per_lk=OUTPUTS_PER_LK_TIME,
                rtol=RTOL, atol_e=ATOL_E):
    cfg = dict(cfg)
    if n_lk_times <= 0 or outputs_per_lk <= 0:
        raise ValueError("Integration duration and output density must be positive.")
    workdir = RUN_DIR / label
    if workdir.exists():
        raise FileExistsError(f"{workdir} exists; use a new label or rerun setup for a new directory.")
    scales = orbital_timescales(cfg)
    n_intervals = max(2, int(np.ceil(n_lk_times * outputs_per_lk)))
    end_yr = n_lk_times * scales["t_LK_yr"]
    targets = np.linspace(0.0, end_yr, n_intervals + 1)
    records = []
    solver = None
    start = perf_counter()
    print(f"{label}: starting SecularMultiple, order={cfg['order']}, "
          f"I0={cfg['mutual_inclination_deg']:g} deg", flush=True)
    with working_directory(workdir):
        try:
            solver = SecularMultiple(**WORKER_OPTIONS)
            configure_secular_solver(solver, cfg["order"], rtol, atol_e)
            initial = make_hierarchy(cfg)
            remote = solver.particles.add_particles(initial)

            
            # Copy the physical properties, child links, and orbital elements once.
            initial.new_channel_to(solver.particles).copy()

            # Confirm that the worker actually received the physical masses. Had trouble with this
            np.testing.assert_allclose(
                remote[:3].mass.value_in(units.MSun),
                initial[:3].mass.value_in(units.MSun),
                rtol=1e-13,
                atol=0.0,
            )

            # Initialize at the initial epoch and find out the worker status.
            solver.evolve_model(0.0 | units.yr)
            for n, target in enumerate(targets):
                if n > 0:
                    solver.evolve_model(float(target) | units.yr)
                flag = int(solver.flag)
                error_code = int(solver.error_code)
                if flag != 0 or error_code != 0:
                    raise RuntimeError(f"SecularMultiple flag={flag}, error_code={error_code}; "
                                       "inspect the worker result before continuing.")
                actual = float(solver.model_time.value_in(units.yr))
                if not np.isclose(actual, target, rtol=1e-11, atol=1e-7):
                    raise RuntimeError(f"Solver returned at {actual} yr instead of {target} yr.")
                records.append(read_secular_state(solver, remote))
                if n == 0:
                    measured = records[0]["mutual_inclination_deg"]
                    if not np.isclose(measured, cfg["mutual_inclination_deg"], atol=1e-5, rtol=0):
                        raise RuntimeError(f"Initial mutual inclination is {measured}, not the requested value.")
                    print(f"  hierarchy committed; initial I = {measured:.6f} deg", flush=True)
        except BaseException:
            if records:
                partial = {key: np.array([row[key] for row in records]) for key in records[0]}
                np.savez_compressed(workdir / "incomplete_history.npz", **partial)
            raise
        finally:
            if solver is not None:
                try:
                    solver.stop()
                except Exception as exc:
                    warnings.warn(f"Worker cleanup failed: {exc!r}")

    history = {key: np.array([row[key] for row in records]) for key in records[0]}
    elapsed = perf_counter() - start
    np.savez_compressed(workdir / "history.npz", **history)
    metadata = {
        "status": "completed_fixed_mass_benchmark", "configuration": cfg,
        "timescales": scales, "number_LK_times": n_lk_times,
        "outputs_per_LK_time": outputs_per_lk, "rtol": rtol, "atol_e": atol_e,
        "wall_seconds": elapsed, "energy_unit": "MSun AU^2 yr^-2",
        "not_a_stellar_survival_simulation": True,
    }
    save_json(workdir / "run.json", metadata)
    print(f"  done; e_max(sampled)={history['e_in'].max():.6f}; elapsed={elapsed:.2f} s", flush=True)
    return history, metadata

## 4. Short worker/element-hierarchy check

This is where we test initialization, particle linking, evolution, and data retrieval.

In [ ]:
smoke_history, smoke_metadata = run_secular(BASE, "00_worker_check", n_lk_times=0.01, outputs_per_lk=200)
print("Worker/hierarchy check completed.")
print("Last integration time [yr]:", smoke_history["time_yr"][-1])
print("Last inner eccentricity:", smoke_history["e_in"][-1])

## 5. Inclination series at quadrupole order

For a nearly circular inner planet, gravitational interactions set a natural limit on how stretched out its orbit can become:

$
e_{\max}\simeq\sqrt{1-\frac53\cos^2 I_0} 
$

This is from Naoz (2016), written in 2.2.1

If the initial orbital tilt ($I_0$) falls outside this specific window, large changes in the orbit's shape won't happen.


Because our planet has a small mass, this formula can act as a good close approximation. **Do make sure your simulation runs long enough to capture the full peak**. It did happen that I got way off values because I was using too short runs. 

This is just to test our code works properly.


In [ ]:
benchmark_histories = {}
benchmark_metadata = {}
for inclination in INCLINATIONS_DEG:
    cfg = {**BASE, "mutual_inclination_deg": float(inclination)}
    label = f"quadrupole_I{int(inclination):03d}"
    history, metadata = run_secular(cfg, label)
    benchmark_histories[inclination] = history
    benchmark_metadata[inclination] = metadata

plot_paths = []
fig, ax = plt.subplots(figsize=(9, 5))
for inclination, history in benchmark_histories.items():
    ax.plot(history["time_yr"] / scales["t_LK_yr"], history["e_in"], label=f"I0 = {inclination:g} deg")
ax.set_xlabel("Integration time / characteristic LK time")
ax.set_ylabel("Inner eccentricity")
ax.set_title("Fixed-mass quadrupole benchmark")
ax.legend(frameon=False, ncols=2)
fig.tight_layout()
path = PLOT_DIR / "quadrupole_eccentricity_histories.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plot_paths.append(path)
plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
grid = np.linspace(0, 180, 721)
ax.plot(grid, [quadrupole_emax_limit(i) for i in grid], linestyle="--", label="Near-circular test-particle limit")
ax.plot(list(benchmark_histories), [h["e_in"].max() for h in benchmark_histories.values()],
        linestyle="none", marker="o", label="AMUSE: sampled maxima")
ax.set_xlabel("Initial mutual inclination [deg]")
ax.set_ylabel("Maximum inner eccentricity")
ax.legend(frameon=False)
fig.tight_layout()
path = PLOT_DIR / "quadrupole_emax_check.png"
fig.savefig(path, dpi=180, bbox_inches="tight")
plot_paths.append(path)
plt.show()

## 6. Conservation checks and numerical resolution tests

In this type of calculation, the semimajor axes and the total orbital momentum vector should stay constant, and the secular Hamiltonian (the system's total energy) should be preserved with some room for computer precision. However, this does not mean the planet's eccentricity or its individual momentum stays constant.

For each orbit, the angular momentum vector is calculated as:

$
\mathbf L=\mu\sqrt{GM a(1-e^2)}\,\hat{\mathbf h},\qquad
\hat{\mathbf h}=(\sin i\sin\Omega,-\sin i\cos\Omega,\cos i).
$

The outer orbit uses the total inner mass as one component. We also report the approximate test-particle value `sqrt(1-e_in**2)*cos(I_mutual)` but do not demand exact conservation of it for a finite-mass planet.

Next, we rerun the 60-degree test case with twice as many data points saved, and then run it again with stricter internal calculation tolerances at that same output density. Just because two different output intervals give the same result doesn't automatically mean the solver is accurate (I learnt this the hard way back in undergrad -Kostas).

In [ ]:
def total_orbital_angular_momentum(history, cfg):
    m1, m2, m3 = cfg["m_star_MSun"], cfg["m_planet_MSun"], cfg["m_bd_MSun"]
    total = np.zeros((len(history["time_yr"]), 3))
    for suffix, left, right in (("in", m1, m2), ("out", m1+m2, m3)):
        reduced_mass = left*right/(left+right)
        magnitude = reduced_mass * np.sqrt(G_AU_MSUN_YR*(left+right)*history[f"a_{suffix}_AU"] *
                                           (1-history[f"e_{suffix}"]**2))
        inc, node = history[f"i_{suffix}_rad"], history[f"node_{suffix}_rad"]
        normal = np.column_stack((np.sin(inc)*np.sin(node), -np.sin(inc)*np.cos(node), np.cos(inc)))
        total += magnitude[:, None]*normal
    return total


def dynamics_summary(history, cfg, label):
    L = total_orbital_angular_momentum(history, cfg)
    Lscale = np.linalg.norm(L[0])
    H = history["hamiltonian"]
    if not np.isfinite(H).all():
        raise ValueError("Non-finite Hamiltonian in the saved history.")
    if H[0] == 0.0:
        raise ValueError(
            "This benchmark requires a nonzero initial Hamiltonian "
            "for relative-error normalization."
        )

    Hscale = abs(H[0])
    jz = np.sqrt(1-history["e_in"]**2) * np.cos(np.deg2rad(history["mutual_inclination_deg"]))
    return {
        "run": label, "initial_mutual_inclination_deg": cfg["mutual_inclination_deg"],
        "sampled_e_max": float(history["e_in"].max()),
        "sampled_q_min_AU": float(history["q_in_AU"].min()),
        "analytic_test_particle_emax": quadrupole_emax_limit(cfg["mutual_inclination_deg"]),
        "max_relative_a_in_change": float(np.max(np.abs(history["a_in_AU"]/history["a_in_AU"][0]-1))),
        "max_relative_a_out_change": float(np.max(np.abs(history["a_out_AU"]/history["a_out_AU"][0]-1))),
        "max_relative_total_L_vector_change": float(np.max(np.linalg.norm(L-L[0], axis=1)/Lscale)),
        "max_scaled_H_change": float(np.max(np.abs(H-H[0]))/Hscale),
        "max_absolute_test_particle_jz_change": float(np.max(np.abs(jz-jz[0]))),
    }


summary_rows = []
for inclination, history in benchmark_histories.items():
    cfg = benchmark_metadata[inclination]["configuration"]
    summary_rows.append(dynamics_summary(history, cfg, f"I{inclination:g}"))
write_rows_csv(RUN_DIR / "quadrupole_benchmark_summary.csv", summary_rows)
print(f"{'I0':>7} {'e_max':>10} {'analytic':>10} {'q_min[AU]':>12} {'delta L/L':>12} {'delta H/H':>12}")
for row in summary_rows:
    print(f"{row['initial_mutual_inclination_deg']:7.1f} {row['sampled_e_max']:10.6f} "
          f"{row['analytic_test_particle_emax']:10.6f} {row['sampled_q_min_AU']:12.6f} "
          f"{row['max_relative_total_L_vector_change']:12.3g} {row['max_scaled_H_change']:12.3g}")

base60 = benchmark_histories[60.0]
dense60, dense_metadata = run_secular(BASE, "resolution_60_dense", outputs_per_lk=2*OUTPUTS_PER_LK_TIME)
tight60, tight_metadata = run_secular(
    BASE,
    "resolution_60_dense_tight",
    outputs_per_lk=2 * OUTPUTS_PER_LK_TIME,
    rtol=1e-14,
    atol_e=1e-16,
)

# The denser run should contain every time in the default run
np.testing.assert_allclose(
    base60["time_yr"],
    dense60["time_yr"][::2],
    rtol=1e-12,
    atol=1e-7,
)

# The tolerance comparison must also use identical sample times
np.testing.assert_allclose(
    dense60["time_yr"],
    tight60["time_yr"],
    rtol=1e-12,
    atol=1e-7,
)

resolution_result = {
    "delta_sampled_emax_output_refinement": float(abs(dense60["e_in"].max()-base60["e_in"].max())),
    "delta_sampled_emax_tighter_solver": float(abs(tight60["e_in"].max()-dense60["e_in"].max())),
    "max_e_difference_tighter_solver_common_times": float(np.max(np.abs(tight60["e_in"]-dense60["e_in"]))),
    "max_e_difference_output_refinement_common_times": float(
    np.max(np.abs(dense60["e_in"][::2] - base60["e_in"]))
),
}

# These are not universal guarantees of correctness.
checks = {
    "coplanar_remains_nearly_circular": bool(benchmark_histories[0.0]["e_in"].max() < 0.01),
    "below_LK_window_remains_nearly_circular": bool(benchmark_histories[30.0]["e_in"].max() < 0.01),
    "inclined_emax_matches_limit": bool(all(
        abs(row["sampled_e_max"]-row["analytic_test_particle_emax"]) < 0.005
        for row in summary_rows if row["initial_mutual_inclination_deg"] in (45, 60, 75, 120)
    )),
    "semimajor_axes_constant": bool(all(max(row["max_relative_a_in_change"], row["max_relative_a_out_change"]) < 1e-8
                                        for row in summary_rows)),
    "total_angular_momentum_conserved": bool(all(row["max_relative_total_L_vector_change"] < 1e-7 for row in summary_rows)),
    "secular_hamiltonian_conserved": bool(all(row["max_scaled_H_change"] < 1e-7 for row in summary_rows)),
    "output_extrema_stable": resolution_result["delta_sampled_emax_output_refinement"] < 5e-4,
    "tolerance_extrema_stable": resolution_result["delta_sampled_emax_tighter_solver"] < 5e-4,
    "tolerance_trajectory_stable": resolution_result["max_e_difference_tighter_solver_common_times"] < 1e-3,
    "output_trajectory_stable": (
    resolution_result["max_e_difference_output_refinement_common_times"]
    < 1e-3
),
}

refined_summaries = [
    dynamics_summary(dense60, BASE, "resolution_60_dense"),
    dynamics_summary(tight60, BASE, "resolution_60_dense_tight"),
]

write_rows_csv(
    RUN_DIR / "refined_run_conservation.csv",
    refined_summaries,
)

checks["refined_runs_also_conserve"] = all(
    row["max_relative_a_in_change"] < 1e-8
    and row["max_relative_a_out_change"] < 1e-8
    and row["max_relative_total_L_vector_change"] < 1e-7
    and row["max_scaled_H_change"] < 1e-7
    for row in refined_summaries
)


for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'REVIEW'}")
print("Resolution differences:", resolution_result)
save_json(RUN_DIR / "quadrupole_validation_status.json", {
    "checks": checks, "all_working_checks_pass": bool(all(checks.values())),
    "resolution": resolution_result,
    "direct_Nbody_comparison": "not yet run unless optional section below is enabled",
    "scope": "fixed-mass, conservative quadrupole test; not the coupled survival model",
})

## 7. Target-like orbital scale

**Disabled by default.** Can be enabled by `RUN_TARGET_SCALE_DEMO` 

We reduce `a_out` to 10 AU and increase `e_out` to 0.3, then compare quadrupole and octupole secular evolutions. The initial outer-pericentre/inner-semimajor-axis ratio is 7.

This is mostly to test the importance of expansion order. Its for sure not a reliable engulfment result.

In [ ]:
target_results = {}
if RUN_TARGET_SCALE_DEMO:
    if not all(checks.values()):
        raise RuntimeError("Review the conservative benchmark before proceeding to the target-scale demo.")
    for order in ("quadrupole", "octupole"):
        cfg = {**BASE, "a_out_AU": 10.0, "e_out": 0.3, "order": order}
        history, metadata = run_secular(cfg, f"target_scale_{order}")
        target_results[order] = history
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for order, history in target_results.items():
        ax.plot(history["time_yr"]/1000, history["e_in"], label=order)
    ax.set_xlabel("Integration time [thousand years]")
    ax.set_ylabel("Inner eccentricity")
    ax.set_title("Illustrative target-scale case: not yet a survival model")
    ax.legend(frameon=False)
    fig.tight_layout()
    path = PLOT_DIR / "target_scale_order_comparison.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plot_paths.append(path)
    plt.show()
else:
    print("Target-scale demo skipped. Keep it off for the first pass.")

## 8. Direct N-body validation with Huayno

**Disabled by default.** This is an additional, expensive check. It uses the same  `a_out = 20 AU` configuration as the 60-degree benchmark, starting at a larger inner eccentricity of 0.01 to shorten the initial near-circular growth interval. We compare a finite interval of five characteristic LK times, not a full stellar lifetime (for now!).

The secular comparison includes pair terms through fifth order; the exact N-body dynamics has no such truncation.

In [ ]:
def rotation_matrix(inc, omega, node):
    def rz(angle):
        c, s = np.cos(angle), np.sin(angle)
        return np.array([[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]])
    c, s = np.cos(inc), np.sin(inc)
    rx = np.array([[1.0, 0.0, 0.0], [0.0, c, -s], [0.0, s, c]])
    return rz(node) @ rx @ rz(omega)


def kepler_relative_state(a, e, inc, omega, node, true_anomaly, total_mass):
    """Return relative position [AU] and velocity [AU/yr] at an anomaly."""
    p = a*(1-e*e)
    r = p/(1+e*np.cos(true_anomaly))
    r_pf = r*np.array([np.cos(true_anomaly), np.sin(true_anomaly), 0.0])
    v_pf = np.sqrt(G_AU_MSUN_YR*total_mass/p)*np.array([-np.sin(true_anomaly), e+np.cos(true_anomaly), 0.0])
    rotation = rotation_matrix(inc, omega, node)
    return rotation @ r_pf, rotation @ v_pf


def osculating_elements(r, v, total_mass):
    """Return semimajor axis, eccentricity and unit orbit normal from relative vectors."""
    mu = G_AU_MSUN_YR*total_mass
    distance = np.linalg.norm(r)
    h = np.cross(r, v)
    hnorm = np.linalg.norm(h)
    energy = 0.5*np.dot(v, v)-mu/distance
    if energy >= 0 or hnorm <= 0:
        raise RuntimeError("Unbound or radial orbit encountered; stop this validation case.")
    eccentricity_vector = np.cross(v, h)/mu-r/distance
    return -mu/(2*energy), np.linalg.norm(eccentricity_vector), h/hnorm


def cartesian_initial_conditions(cfg, f_inner_deg=0.0, f_outer_deg=73.0):
    m1, m2, m3 = cfg["m_star_MSun"], cfg["m_planet_MSun"], cfg["m_bd_MSun"]
    m12, mtot = m1+m2, m1+m2+m3
    inc_out = np.deg2rad(cfg["outer_reference_inclination_deg"])
    inc_in = inc_out+np.deg2rad(cfg["mutual_inclination_deg"])
    r_in, v_in = kepler_relative_state(cfg["a_in_AU"], cfg["e_in"], inc_in,
        np.deg2rad(cfg["omega_in_deg"]), np.deg2rad(cfg["node_in_deg"]), np.deg2rad(f_inner_deg), m12)
    r_out, v_out = kepler_relative_state(cfg["a_out_AU"], cfg["e_out"], inc_out,
        np.deg2rad(cfg["omega_out_deg"]), np.deg2rad(cfg["node_out_deg"]), np.deg2rad(f_outer_deg), mtot)
    # r_in = r_planet - r_star; r_out = r_BD - r_inner_COM.
    r_com_inner = -(m3/mtot)*r_out
    v_com_inner = -(m3/mtot)*v_out
    pos = np.vstack((r_com_inner-(m2/m12)*r_in, r_com_inner+(m1/m12)*r_in, (m12/mtot)*r_out))
    vel = np.vstack((v_com_inner-(m2/m12)*v_in, v_com_inner+(m1/m12)*v_in, (m12/mtot)*v_out))
    masses = np.array([m1, m2, m3])
    return masses, pos, vel


def cartesian_orbit_diagnostics(masses, pos, vel):
    m1, m2, m3 = masses
    m12 = m1+m2
    r_inner = pos[1]-pos[0]
    v_inner = vel[1]-vel[0]
    r_outer = pos[2]-(m1*pos[0]+m2*pos[1])/m12
    v_outer = vel[2]-(m1*vel[0]+m2*vel[1])/m12
    a_in, e_in, n_in = osculating_elements(r_inner, v_inner, m12)
    a_out, e_out, n_out = osculating_elements(r_outer, v_outer, m12+m3)
    I = np.rad2deg(np.arccos(np.clip(np.dot(n_in, n_out), -1, 1)))
    return {"a_in_AU": float(a_in), "e_in": float(e_in), "a_out_AU": float(a_out),
            "e_out": float(e_out), "mutual_inclination_deg": float(I)}


# This geometry test does not start a gravity worker.
masses_test, pos_test, vel_test = cartesian_initial_conditions(BASE)
initial_check = cartesian_orbit_diagnostics(masses_test, pos_test, vel_test)
for key in ("a_in_AU", "a_out_AU", "e_in", "e_out", "mutual_inclination_deg"):
    assert np.isclose(initial_check[key], BASE[key], rtol=1e-9, atol=1e-10), (key, initial_check[key], BASE[key])
assert np.linalg.norm(np.sum(masses_test[:, None]*pos_test, axis=0)/masses_test.sum()) < 1e-12
assert np.linalg.norm(np.sum(masses_test[:, None]*vel_test, axis=0)) < 1e-12
print("Jacobi-to-Cartesian round-trip checks passed.")

In [ ]:
def run_direct(cfg, label, eta, n_lk_times=5.0, n_outputs=1001):
    from amuse.community.huayno import Huayno
    from amuse.units import nbody_system

    masses, positions, velocities = cartesian_initial_conditions(cfg)
    particles = Particles(3)
    particles.mass = masses | units.MSun
    particles.position = positions | units.au
    particles.velocity = velocities | (units.au / units.yr)
    particles.radius = 0.0 | units.au
    converter = nbody_system.nbody_to_si(float(masses.sum()) | units.MSun, cfg["a_in_AU"] | units.au)
    workdir = RUN_DIR / label
    if workdir.exists():
        raise FileExistsError(f"{workdir} already exists")
    targets = np.linspace(0, n_lk_times*orbital_timescales(cfg)["t_LK_yr"], n_outputs)
    rows = []
    solver = None
    started = perf_counter()
    with working_directory(workdir):
        try:
            solver = Huayno(converter, **WORKER_OPTIONS)
            solver.parameters.epsilon_squared = 0.0 | units.au**2
            solver.parameters.timestep_parameter = eta
            solver.parameters.inttype_parameter = Huayno.inttypes.CC_KEPLER
            solver.particles.add_particles(particles)
            channel = solver.particles.new_channel_to(particles)
            for j, target in enumerate(targets):
                if j > 0:
                    solver.evolve_model(float(target) | units.yr)
                channel.copy_attributes(["x", "y", "z", "vx", "vy", "vz"])
                row = cartesian_orbit_diagnostics(masses, particles.position.value_in(units.au),
                                                  particles.velocity.value_in(units.au/units.yr))
                row["time_yr"] = float(solver.model_time.value_in(units.yr))
                if not np.isclose(row["time_yr"], target, rtol=1e-10, atol=1e-7):
                    raise RuntimeError("Huayno did not reach requested time.")
                row["energy"] = float((solver.kinetic_energy+solver.potential_energy).value_in(ENERGY_UNIT))
                rows.append(row)
                if j % 200 == 0:
                    print(f"  {label}: {100*j/(len(targets)-1):.0f}%", flush=True)
        except BaseException:
            if rows:
                partial = {key: np.array([row[key] for row in rows]) for key in rows[0]}
                np.savez_compressed(workdir / "incomplete_history.npz", **partial)
            raise
        finally:
            if solver is not None:
                try:
                    solver.stop()
                except Exception as exc:
                    warnings.warn(f"Worker cleanup failed: {exc!r}")
    history = {key: np.array([row[key] for row in rows]) for key in rows[0]}
    np.savez_compressed(workdir / "history.npz", **history)
    save_json(workdir / "run.json", {"configuration": cfg, "eta": eta, "wall_seconds": perf_counter()-started,
                                     "number_LK_times": n_lk_times, "n_outputs": n_outputs,
                                     "integrator": "CC_KEPLER", "initial_true_anomalies_deg": [0.0, 73.0]})
    return history


direct_status = {"status": "not_run", "reason": "RUN_DIRECT_VALIDATION=False"}
if RUN_DIRECT_VALIDATION:
    if not all(checks.values()):
        raise RuntimeError("Review the secular benchmark before the optional direct calculation.")
    direct_cfg = {**BASE, "e_in": 0.01, "order": "fifth"}
    direct_secular, _ = run_secular(direct_cfg, "direct_reference_secular", n_lk_times=5, outputs_per_lk=200)
    direct_loose = run_direct(direct_cfg, "direct_huayno_eta001", eta=0.01)
    direct_tight = run_direct(direct_cfg, "direct_huayno_eta0005", eta=0.005)
    energy_error = lambda h: float(np.max(np.abs((h["energy"]-h["energy"][0])/h["energy"][0])))
    direct_status = {
        "status": "computed_requires_comparison",
        "max_total_energy_error_eta001": energy_error(direct_loose),
        "max_total_energy_error_eta0005": energy_error(direct_tight),
        "emax_eta001": float(direct_loose["e_in"].max()),
        "emax_eta0005": float(direct_tight["e_in"].max()),
        "emax_secular": float(direct_secular["e_in"].max()),
        "max_e_difference_between_direct_settings": float(np.max(np.abs(direct_loose["e_in"]-direct_tight["e_in"]))),
        "note": "Direct values are osculating; do not require pointwise identity with secular output.",
    }
    print(direct_status)
    fig, ax = plt.subplots(figsize=(9, 5))
    for label, history in (("Secular, fifth order", direct_secular), ("Huayno eta=0.01", direct_loose),
                           ("Huayno eta=0.005", direct_tight)):
        ax.plot(history["time_yr"]/1000, history["e_in"], label=label)
    ax.set_xlabel("Integration time [thousand years]")
    ax.set_ylabel("Inner eccentricity")
    ax.legend(frameon=False)
    fig.tight_layout()
    path = PLOT_DIR / "direct_secular_comparison.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plot_paths.append(path)
    plt.show()
else:
    print("Direct N-body integration skipped. Analytical checks are not a substitute for this later comparison.")
save_json(RUN_DIR / "direct_validation_status.json", direct_status)

In [ ]:

# This is a later addition so I didn't have time to make it optional through code but probably better to run it if you have already ran the direct N-body comparison. 

# Expansion order comparison at the closer orbital scale
target_fifth_cfg = {
    **BASE,
    "a_out_AU": 10.0,
    "e_out": 0.3,
    "order": "fifth",
}

target_fifth, target_fifth_metadata = run_secular(
    target_fifth_cfg,
    "target_scale_fifth",
)
target_results["fifth"] = target_fifth

# A third accuracy setting for comparison with the direct N-body integration. The eta=0.0025 run is not expected to be identical to the eta=0.005 run, but it should be very close. 
direct_finer = run_direct(
    direct_cfg,
    "direct_huayno_eta0p0025",
    eta=0.0025,
    n_lk_times=5.0,
    n_outputs=len(direct_tight["time_yr"]),
)

np.testing.assert_allclose(
    direct_finer["time_yr"],
    direct_tight["time_yr"],
    rtol=1e-12,
    atol=1e-7,
)

print("Finer direct e_max:", direct_finer["e_in"].max())
print(
    "Maximum eccentricity difference from eta=0.005:",
    np.max(np.abs(direct_finer["e_in"] - direct_tight["e_in"])),
)